<a href="https://colab.research.google.com/github/Lindart0312/machine-learning-practice/blob/main/06_cross_validation_hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 교차 검증과 그리드 서치

## 검증 데이터셋

In [5]:
import pandas as pd

wine = pd.read_csv('https://bit.ly/wine-date')

### 문제 1 : wine 데이터 확인

In [6]:
# wine 처음 5개 행 데이터 확인
wine.head(5)

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [10]:
# wine 전체 행의 개수 확인
print(wine.shape)

(6497, 4)


In [54]:
# wine 데이터 통계값 확인 (각 특성별 평균, 표준편차, 최소값, 최대값 등)
wine.describe()

,alcohol,sugar,pH,class
count,6497.000000,6497.000000,6497.000000,6497.000000
mean,10.491801,5.443235,3.218501,0.753886
std,1.192712,4.757804,0.160787,0.430779
min,8.000000,0.600000,2.720000,0.000000
25%,9.500000,1.800000,3.110000,1.000000
50%,10.300000,3.000000,3.210000,1.000000
75%,11.300000,8.100000,3.320000,1.000000
max,14.900000,65.800000,4.010000,1.000000


In [12]:
# 화이트 와인, 레드 와인 데이터 개수 확인
wine['class'].value_counts()

,count
class,
1.0,4898
0.0,1599


### 데이터셋 분류

In [13]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [14]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42)

In [15]:
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42)

In [16]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [17]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


## 교차 검증

In [18]:
from sklearn.model_selection import cross_validate

scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.01103139, 0.00975561, 0.00910234, 0.00873494, 0.00838566]), 'score_time': array([0.00161886, 0.00127292, 0.00128603, 0.00117302, 0.00116587]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [19]:
import numpy as np

print(np.mean(scores['test_score']))

0.855300214703487


In [20]:
from sklearn.model_selection import StratifiedKFold

scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


In [21]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


## 하이퍼파라미터 튜닝

In [22]:
from sklearn.model_selection import GridSearchCV

params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [23]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

In [24]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [25]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [26]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [27]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [28]:
best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


In [29]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001), # 그리드서치, 단위로 써있음 그대로 쓰라는거
          'max_depth': range(5, 20, 1), # 5에서 20까지 1씩 올라감
          'min_samples_split': range(2, 100, 10)
          }

In [30]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [37]:
print(gs.best_params_)

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}


In [32]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8683865773302731


In [33]:
# 교차검증 수행 시간 프린트
gs.cv_results_['mean_fit_time']

array([0.00762067, 0.0079052 , 0.00755143, ..., 0.01099253, 0.01300435,
       0.01140175])

### 랜덤 서치

In [34]:
from scipy.stats import uniform, randint

In [35]:
# 균등 분포 샘플링
rgen = randint(0, 10)
rgen.rvs(10)

array([7, 5, 4, 1, 7, 5, 1, 7, 9, 4])

In [36]:
np.unique(rgen.rvs(1000), return_counts=True) # 빈도도 함께 출력

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([ 96,  94, 124, 101, 114,  99,  87,  99,  94,  92]))

In [38]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.12994241, 0.75682809, 0.86533488, 0.60674772, 0.45379095,
       0.98412376, 0.83617637, 0.95712647, 0.20458715, 0.62093862])

In [39]:
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25),
          }

In [40]:
from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb8cb955730>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7bb8cb93d760>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb8cb93e5a0>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb8cb93f1d0>},
                   random_state=42)

In [41]:
print(rs.best_params_)

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [42]:
print(np.max(rs.cv_results_['mean_test_score']))

0.8695428296438884


In [43]:
dt = rs.best_estimator_

print(dt.score(test_input, test_target))

0.86


In [44]:
rs.cv_results_['mean_fit_time']

array([0.00846128, 0.00769138, 0.00800939, 0.0086607 , 0.00757003,
       0.00731578, 0.00686803, 0.00706472, 0.00755587, 0.0080524 ,
       0.00754571, 0.00693936, 0.00713139, 0.00777917, 0.00723209,
       0.00787702, 0.00693989, 0.01271038, 0.00858912, 0.00729179,
       0.00904713, 0.00757046, 0.00710969, 0.00816064, 0.00998721,
       0.01251121, 0.00763259, 0.01572952, 0.01237831, 0.00985894,
       0.0072444 , 0.01022024, 0.0083437 , 0.00985265, 0.01181355,
       0.01389937, 0.01551347, 0.01499796, 0.00683708, 0.00771441,
       0.01384931, 0.01621423, 0.00689821, 0.0074111 , 0.01726623,
       0.01637888, 0.01493959, 0.01556926, 0.01686563, 0.01639919,
       0.01583619, 0.0162746 , 0.01412344, 0.01350846, 0.01468582,
       0.01690378, 0.01554561, 0.01584735, 0.01349607, 0.00876384,
       0.02040176, 0.01479549, 0.00673676, 0.00889587, 0.00885262,
       0.01694665, 0.01351595, 0.01679225, 0.01405029, 0.01607461,
       0.01318345, 0.0174686 , 0.01494269, 0.01372919, 0.01685

In [45]:
print(np.mean(rs.cv_results_['mean_fit_time']))

0.012076275825500488


### 결정트리 분할 옵션 변경

In [46]:
rs2 = RandomizedSearchCV(DecisionTreeClassifier(splitter='random', random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
rs2.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42,
                                                    splitter='random'),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb8cb955730>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7bb8cb93d760>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb8cb93e5a0>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7bb8cb93f1d0>},
                   random_state=42)

In [47]:
print(rs2.best_params_)
print(np.max(rs2.cv_results_['mean_test_score']))

dt = rs2.best_estimator_
print(dt.score(test_input, test_target))

{'max_depth': 43, 'min_impurity_decrease': np.float64(0.00011407982271508446), 'min_samples_leaf': 19, 'min_samples_split': 18}
0.8458726956392981
0.786923076923077


In [48]:
rs2.cv_results_['mean_fit_time']

array([0.00612764, 0.00675988, 0.00388331, 0.00504084, 0.00637684,
       0.00787616, 0.00658979, 0.00653586, 0.00729785, 0.00694232,
       0.00751104, 0.00627689, 0.00666795, 0.0078577 , 0.00389771,
       0.00655828, 0.00778613, 0.00864239, 0.00648155, 0.00339851,
       0.00415411, 0.00774517, 0.00814514, 0.00598879, 0.00654216,
       0.00658402, 0.00344009, 0.00329466, 0.00695572, 0.00715446,
       0.0066299 , 0.00786157, 0.00629034, 0.00815687, 0.00806441,
       0.009273  , 0.0075757 , 0.00702467, 0.00840335, 0.00653243,
       0.00672617, 0.0077333 , 0.00702038, 0.0078321 , 0.00804253,
       0.00659752, 0.00781078, 0.00831985, 0.01035886, 0.00726652,
       0.00816574, 0.00343328, 0.00320401, 0.00592947, 0.00652971,
       0.01096001, 0.00864592, 0.00705137, 0.00793643, 0.00668845,
       0.00717945, 0.00693398, 0.00476089, 0.0071691 , 0.00685372,
       0.00333004, 0.00332799, 0.00894475, 0.00764351, 0.00667706,
       0.00707154, 0.00606623, 0.00652518, 0.00643926, 0.00533

In [49]:
print(np.mean(rs2.cv_results_['mean_fit_time']))

0.006666752338409424


문제 2 : 위 코드가 기존 랜덤 서치 코드와 다른 점을 2가지 적어보세요.
- 결정코드는 하이퍼파라미터 탐색을 초기에 디씨젼트리를 만들어서 특정값(random_state(42))을 만듦
- 랜덤 서치는 최적의 모델을 기계가 자동적으로 찾아줌

-  랜덤 서치는 매개 변수 조건이 많으면 실행 속도가 빨라지고, 결정트리는 노드를 랜덤하게 분할하기에 계산속도가 빠름